In [1]:
import torch

def evaluate_feature_sparsity(
    model,
    dataloader,
    layer_path,
    device="cuda",
    eps=1e-3,
    var_thr=1e-4,
):
    """
    Evaluate feature sparsity and effective feature dimension
    at a given layer.

    Returns:
        dict with:
            sparsity_ratio
            effective_dim
            total_dim
            mean_l1_norm
    """

    model = model.to(device).eval()

    activations = []

    # ---------- hook ----------
    def hook_fn(_, __, output):
        activations.append(output.detach().cpu())

    module = dict(model.named_modules())[layer_path]
    handle = module.register_forward_hook(hook_fn)

    # ---------- forward ----------
    with torch.no_grad():
        for batch in dataloader:
            x = batch[0] if isinstance(batch, (list, tuple)) else batch
            x = x.to(device)
            _ = model(x)

    handle.remove()

    # ---------- build feature matrix ----------
    H = torch.cat(activations, dim=0)

    # flatten if needed
    H = H.view(H.size(0), -1)

    # ---------- metrics ----------

    # feature sparsity (near-zero activations)
    sparsity_ratio = (H.abs() < eps).float().mean().item()

    # effective feature dimension
    feature_var = H.var(dim=0)
    effective_dim = (feature_var > var_thr).float().sum().item()

    total_dim = H.size(1)

    # representation magnitude proxy
    mean_l1_norm = H.abs().mean().item()

    return {
        "sparsity_ratio": sparsity_ratio,
        "effective_dim": effective_dim,
        "total_dim": total_dim,
        "mean_l1_norm": mean_l1_norm,
    }

In [3]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from models import BigCNN, StudentCNN, TeacherCNN
from lth_utils import get_prunable_layers
from training import eval_epoch
from cka_utils import compute_cka_matrix
from mnist1d_dataset import get_mnist1d_loaders, get_dataset_info

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader, test_loader = get_mnist1d_loaders()

info = get_dataset_info()
print(info)

teacher = TeacherCNN().to(device)
teacher.load_state_dict(torch.load("models/best_teacher_model.pth"))

def load_student(ckpt_path, device):
    model = StudentCNN().to(device)

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    return model

num_students = 10
students_paths = [f"models/best_kd_student_{i}.pth" for i in range(num_students)]

students = [
    load_student(path, device)
    for path in students_paths
]

def load_pruned_ticket(ckpt_path, device):
    model = BigCNN().to(device)

    # Apply pruning structure FIRST
    parameters_to_prune = get_prunable_layers(model)
    for m, p in parameters_to_prune:
        prune.identity(m, p)

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    return model

num_tickets = 10
ticket_paths = [f"models/lth/ticket_model_run{i}.pth" for i in range(num_tickets)]

tickets = [
    load_pruned_ticket(path, device)
    for path in ticket_paths
]

dense_model = BigCNN().to(device)
dense_model.load_state_dict(torch.load("models/lth/dense_model_run0.pth", map_location=device))


File already exists. Skipping download.
Successfully loaded data from ./mnist1d_data.pkl
File already exists. Skipping download.
Successfully loaded data from ./mnist1d_data.pkl
{'n_train': 4000, 'n_test': 1000, 'input_length': 40, 'num_classes': 10}


<All keys matched successfully>

In [4]:
stats_teacher = evaluate_feature_sparsity(
    teacher,
    test_loader,
    layer_path="classifier.4",   # penultimate layer example
)

stats_student = evaluate_feature_sparsity(
    students[0],
    test_loader,
    layer_path="classifier.1",   # penultimate layer example
)

stats_dense = evaluate_feature_sparsity(
    dense_model,
    test_loader,
    layer_path="classifier.1",
)

stats_ticket = evaluate_feature_sparsity(
    tickets[0],
    test_loader,
    layer_path="classifier.1",
)

print(f"Teacher: {stats_teacher}")
print(f"Student: {stats_student}")
print(f"Dense: {stats_dense}")
print(f"Ticket: {stats_ticket}")


Teacher: {'sparsity_ratio': 0.12797071039676666, 'effective_dim': 445.0, 'total_dim': 512, 'mean_l1_norm': 0.6484830379486084}
Student: {'sparsity_ratio': 0.08921052515506744, 'effective_dim': 16.0, 'total_dim': 19, 'mean_l1_norm': 4.434450626373291}
Dense: {'sparsity_ratio': 0.0003124999930150807, 'effective_dim': 64.0, 'total_dim': 64, 'mean_l1_norm': 1.9088164567947388}
Ticket: {'sparsity_ratio': 0.0005624999757856131, 'effective_dim': 64.0, 'total_dim': 64, 'mean_l1_norm': 1.3041791915893555}
